# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ² protein mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is specified via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and check the overall description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Display the dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Let's review record sets and their fields in this dataset, referencing each entity by its `@id`.

We will print all record set IDs, their descriptions, and the field IDs and names within each record set.

In [ ]:
# Identify record sets and their fields by `@id`
from mlcroissant.structs.record_set import RecordSet

record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    # For some schemas, 'recordSet' could be a single dict. Try to detect that.
    rs = dataset.metadata.to_json().get('recordSet', None)
    if rs and isinstance(rs, dict):
        record_set_ids = [rs['@id']]

print(f"Record Sets (@id): {record_set_ids}")

if not record_set_ids:
    print("No record sets found in the schema.")
else:
    for rs_id in record_set_ids:
        print(f"--- Record Set @id: {rs_id} ---")
        rs_obj = dataset.metadata.lookup(rs_id)
        print(f"  Name: {getattr(rs_obj, 'name', 'N/A')}")
        print(f"  Description: {getattr(rs_obj, 'description', 'N/A')}")
        fields = getattr(rs_obj, 'fields', [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    Field @id: {field['@id']}  Name: {field.get('name', 'N/A')}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s printed above.

**Note:** For this dataset, record sets are referenced only by their `@id`. Adjust the `record_sets` list as needed based on the actual record set IDs obtained in the previous cell.

In [ ]:
# List your main record set @ids here after running the previous cell.
# For the FAIR^2 dataset, let's find the recordSet IDs from the metadata JSON:
record_sets = [
    # Example: 'https://api.app.sen.science/frontiers/7154140/recordSet/table1'
    # Replace this example with the actual @id(s) from the previous cell output.
]

if not record_sets:
    # Try to auto-detect by looking for tabular files in the distribution
    dist = dataset.metadata.to_json().get('distribution', [])
    if isinstance(dist, dict):
        dist = [dist]
    for d in dist:
        file_id = d.get('@id')
        print(f"Distribution file detected: {file_id}")
        # Try as record_set
        try:
            _ = list(dataset.records(record_set=file_id))
            record_sets.append(file_id)
        except Exception:
            # Not a record set, skip
            pass
    print(f"Inferred record_sets: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    try:
        print(f"Reading records for record_set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Example: print first five rows and columns for one (replace with actual @id from previous outputs)
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by a key attribute.

Replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id`'s or DataFrame column names from your dataset for meaningful analysis.

In [ ]:
import numpy as np

if dataframes:
    # Use the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Sample columns: {df.columns.tolist()}")
    
    # Choose a numeric field for demonstration (assume a typical mass spec field e.g. 'intensity' or 'MolecularWeight')
    # Replace 'MolecularWeight' with the actual numeric column
    numeric_field = None
    for col in df.columns:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            numeric_field = col
            break
    if numeric_field is None:
        # If cannot auto-detect, fallback to manual entry
        numeric_field = df.columns[0]  # Modify as appropriate
    print(f"Using numeric field for analysis: {numeric_field}")

    # Filter based on an arbitrary threshold for the selected numeric field
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Add normalized column
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if present
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No categorical group field detected for grouping.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Let's plot the distribution of a numeric field and, if possible, compare means across grouped categories.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Histogram of numeric field
    if 'numeric_field' in locals() and numeric_field in df:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
    
    # Boxplot by group if available
    if 'group_field' in locals() and group_field and group_field in df:
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No DataFrames to visualize.")

## 6. Conclusion

- We loaded the FAIRˆ² mass spectrometry dataset using Croissant schema and the `mlcroissant` library.
- We explored record sets, their fields, unique identifiers, and loaded records into pandas DataFrames.
- Simple EDA and visualization steps highlighted the utility of Croissant schemas for structured scientific datasets.

You can further adapt this notebook to more specific analyses based on experimental hypotheses or domain requirements.